# 03 — Phase 3: fixed-budget SVAMP adaptation (the outcome variable)

From **each** of the 5 checkpoints, run the *identical* adaptation recipe
(Madhur's framing: fixed-budget future adaptability, everything defined up
front): 256 frozen SVAMP train questions · **50 GRPO updates** · same
optimizer/LR/seed · greedy eval on the same frozen 100 SVAMP questions
before/after + every 10 updates (adaptation-speed curve).

**Choice logged (open question #2):** adaptation algorithm = **GRPO**, keeping
"stage B = RL" apples-to-apples with the proposal. SFT alternative would be a
flag in `src/adaptation.py`.

**GPU:** A100 or L4. 5 runs × 50 updates. Log units per run in `compute_log.md`.

In [ ]:
%pip install -q trl==1.6.0 transformers==5.13.0 datasets==5.0.0 accelerate==1.14.0 pytest==8.4.2 numpy==2.3.5 scipy==1.16.3 pandas==2.3.3 matplotlib==3.10.6

In [ ]:
import json, os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/eaaj-pilot")
else:
    PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

In [ ]:
from src.repro import get_active_run

PILOT = json.loads(Path("pilot_config.json").read_text())
RUN_DIR = get_active_run(PROJECT_DIR)
CKPTS = PILOT["stage_a"]["checkpoint_steps"]
ADAPT_ROOT = RUN_DIR / "adaptation"
ADAPT_ROOT.mkdir(exist_ok=True)
print("adapting checkpoints from:", RUN_DIR)

In [ ]:
from src.adaptation import run_fixed_budget_adaptation

recipe = PILOT["adaptation"]
summaries = []
for n in CKPTS:
    out_dir = ADAPT_ROOT / f"ckpt-{n}"
    summary_path = out_dir / "summary.json"
    if summary_path.exists():
        print(f"ckpt {n}: adaptation already done, skipping")
        summaries.append(json.loads(summary_path.read_text()))
        continue
    s = run_fixed_budget_adaptation(
        checkpoint_path=RUN_DIR / f"ckpt-{n}", out_dir=out_dir,
        budget_updates=recipe["budget_updates"], eval_every=recipe["eval_every"],
        seed=PILOT["seed"], learning_rate=recipe["learning_rate"],
        num_generations=recipe["num_generations"],
        per_device_batch=recipe["per_device_train_batch_size"],
        grad_accum=recipe["gradient_accumulation_steps"],
        beta=recipe["beta"], temperature=recipe["temperature"], top_p=recipe["top_p"],
        max_prompt_length=recipe["max_prompt_length"],
        max_completion_length=recipe["max_completion_length"], bf16=True)
    summaries.append(s)
    print(f"ckpt {n}: {s['acc_before']:.3f} -> {s['acc_after']:.3f} "
          f"(Δ={s['delta_acc']:+.3f}) in {s['wall_seconds']/60:.1f} min")

In [ ]:
import pandas as pd
df = pd.DataFrame(summaries)
df["ckpt"] = [int(Path(c).name.split("-")[1]) for c in df.checkpoint]
print(df[["ckpt", "acc_before", "acc_after", "delta_acc", "wall_seconds"]]
      .sort_values("ckpt").to_string(index=False))
print(">> compute_log.md: record units + screenshot")